In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import re
import sqlite3
import pandas as pd
from datetime import datetime

Mounted at /content/drive


In [2]:
phase4_path = "/content/drive/MyDrive/Capstone Project/partnerlens_synthetic_data_package/partnerlens_data_package/outputs/phase4_sql_baseline"

phase6_path = "/content/drive/MyDrive/Capstone Project/partnerlens_synthetic_data_package/partnerlens_data_package/outputs/phase6_citation_guardrails"

os.makedirs(phase6_path, exist_ok=True)

db_path = os.path.join(phase4_path, "partnerlens.db")

print("Database path:", db_path)
print("Phase 6 output path:", phase6_path)

Database path: /content/drive/MyDrive/Capstone Project/partnerlens_synthetic_data_package/partnerlens_data_package/outputs/phase4_sql_baseline/partnerlens.db
Phase 6 output path: /content/drive/MyDrive/Capstone Project/partnerlens_synthetic_data_package/partnerlens_data_package/outputs/phase6_citation_guardrails


In [3]:
conn = sqlite3.connect(db_path)

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

tables

,name
0,partner_master
1,partner_pricing
2,partner_metrics


In [5]:
def run_sql(query):
    return pd.read_sql_query(query, conn)

In [6]:
run_sql("SELECT * FROM partner_master LIMIT 5;")

,partner_id,partner_name,partner_type,industry_vertical,partner_size,partner_status,country,state,city,region,integration_type,sales_channel,risk_tier,kyc_status,pci_level,onboarding_date,relationship_manager_region,synthetic_record_flag,industry,status
0,P000001,NorthStar Commerce 0001,Technology Platform,Travel,SMB,Suspended,US,NY,New York,Northeast,API,Direct,High,Approved,Level 3,2019-06-15,Northeast,1,Travel,Suspended
1,P000002,Ironwood Market 0002,ISV,SaaS,Mid-Market,Pilot,US,VA,Norfolk,South,SDK,Direct,Low,Approved,Level 4,2020-02-13,South,1,SaaS,Pilot
2,P000003,Evergreen Commerce 0003,Technology Platform,Professional Services,SMB,Active,US,CO,Colorado Springs,West,API,Direct,Low,Approved,Level 3,2023-03-12,West,1,Professional Services,Active
3,P000004,Summit Platform 0004,ISO/Reseller,Gaming,SMB,Active,US,GA,Augusta,South,API,Referral,Medium,Approved,Level 4,2020-02-13,South,1,Gaming,Active
4,P000005,Prairie PayTech 0005,Merchant Acquirer,Healthcare,SMB,Active,US,CA,San Diego,West,API,Direct,High,Approved,Level 4,2022-03-09,West,1,Healthcare,Active


In [7]:
def get_database_schema(conn):
    schema = {}

    table_names = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table';",
        conn
    )["name"].tolist()

    for table in table_names:
        columns_df = pd.read_sql_query(f"PRAGMA table_info({table});", conn)
        schema[table] = columns_df["name"].tolist()

    return schema


database_schema = get_database_schema(conn)

database_schema

{'partner_master': ['partner_id',
  'partner_name',
  'partner_type',
  'industry_vertical',
  'partner_size',
  'partner_status',
  'country',
  'state',
  'city',
  'region',
  'integration_type',
  'sales_channel',
  'risk_tier',
  'kyc_status',
  'pci_level',
  'onboarding_date',
  'relationship_manager_region',
  'synthetic_record_flag',
  'industry',
  'status'],
 'partner_pricing': ['assignment_id',
  'partner_id',
  'pricing_plan_id',
  'recommended_pricing_plan_id',
  'effective_date',
  'expiration_date',
  'review_due_date',
  'negotiated_bps',
  'negotiated_per_txn_fee_usd',
  'monthly_minimum_fee_usd',
  'exception_flag',
  'exception_type',
  'approval_status',
  'approved_by_role',
  'pricing_tier',
  'contract_status'],
 'partner_metrics': ['partner_id',
  'month',
  'txn_count',
  'payment_volume_usd',
  'avg_ticket_usd',
  'txn_growth_pct',
  'volume_growth_pct',
  'chargeback_rate',
  'auth_approval_rate',
  'active_merchants',
  'refunds_usd',
  'net_revenue_usd',
 

In [8]:
query_library = {
    "Q001": {
        "intent": "technology_az_growth_above_20",
        "example_question": "Show me technology partners in Arizona with transaction growth above 20%.",
        "keywords": ["technology", "arizona", "az", "growth", "20"],
        "answer_type": "Filtered list",
        "supported": True,
        "sql": """
            SELECT
                pm.partner_id,
                pm.partner_name,
                pm.partner_type,
                pm.industry_vertical,
                pm.state,
                pm.partner_status,
                mt.transaction_month,
                mt.growth_pct
            FROM partner_master pm
            JOIN partner_metrics mt
                ON pm.partner_id = mt.partner_id
            WHERE pm.partner_type LIKE '%Technology%'
              AND pm.state = 'AZ'
              AND mt.growth_pct > 20
            ORDER BY mt.growth_pct DESC;
        """,
        "citations": [
            "partner_master.partner_id",
            "partner_master.partner_name",
            "partner_master.partner_type",
            "partner_master.industry_vertical",
            "partner_master.state",
            "partner_metrics.transaction_month",
            "partner_metrics.growth_pct"
        ]
    },

    "Q002": {
        "intent": "strategic_pricing_partners",
        "example_question": "Which partners are on the Strategic pricing tier?",
        "keywords": ["strategic", "pricing", "tier", "partners"],
        "answer_type": "Filtered list",
        "supported": True,
        "sql": """
            SELECT
                pm.partner_id,
                pm.partner_name,
                pm.partner_status,
                pp.pricing_tier,
                pp.recommended_pricing_plan_id,
                pp.approval_status
            FROM partner_master pm
            JOIN partner_pricing pp
                ON pm.partner_id = pp.partner_id
            WHERE pp.pricing_tier = 'STRATEGIC'
            ORDER BY pm.partner_name;
        """,
        "citations": [
            "partner_master.partner_id",
            "partner_master.partner_name",
            "partner_master.partner_status",
            "partner_pricing.pricing_tier",
            "partner_pricing.recommended_pricing_plan_id",
            "partner_pricing.approval_status"
        ]
    },

    "Q003": {
        "intent": "pricing_reviews_due_90_days",
        "example_question": "Which partners have pricing reviews due in the next 90 days?",
        "keywords": ["pricing", "review", "due", "90", "days"],
        "answer_type": "Filtered list",
        "supported": True,
        "sql": """
            SELECT
                pm.partner_id,
                pm.partner_name,
                pp.pricing_tier,
                pp.approval_status,
                pp.review_due_date,
                pp.contract_status
            FROM partner_master pm
            JOIN partner_pricing pp
                ON pm.partner_id = pp.partner_id
            WHERE date(pp.review_due_date) BETWEEN date('2026-06-28') AND date('2026-09-26')
            ORDER BY date(pp.review_due_date);
        """,
        "citations": [
            "partner_master.partner_id",
            "partner_master.partner_name",
            "partner_pricing.pricing_tier",
            "partner_pricing.approval_status",
            "partner_pricing.review_due_date",
            "partner_pricing.contract_status"
        ]
    },

    "Q004": {
        "intent": "highest_average_transaction_volume_by_industry",
        "example_question": "Which industry has the highest average transaction volume?",
        "keywords": ["industry", "highest", "average", "transaction", "volume"],
        "answer_type": "Ranking summary",
        "supported": True,
        "sql": """
            SELECT
                pm.industry_vertical,
                ROUND(AVG(mt.transaction_volume), 2) AS avg_transaction_volume
            FROM partner_master pm
            JOIN partner_metrics mt
                ON pm.partner_id = mt.partner_id
            GROUP BY pm.industry_vertical
            ORDER BY avg_transaction_volume DESC;
        """,
        "citations": [
            "partner_master.industry_vertical",
            "partner_metrics.transaction_volume"
        ]
    },

    "Q005": {
        "intent": "top_5_partners_by_revenue",
        "example_question": "Show the top 5 partners by revenue.",
        "keywords": ["top", "5", "partners", "revenue"],
        "answer_type": "Top 5 ranking",
        "supported": True,
        "sql": """
            SELECT
                pm.partner_id,
                pm.partner_name,
                ROUND(SUM(mt.revenue), 2) AS total_revenue
            FROM partner_master pm
            JOIN partner_metrics mt
                ON pm.partner_id = mt.partner_id
            GROUP BY pm.partner_id, pm.partner_name
            ORDER BY total_revenue DESC
            LIMIT 5;
        """,
        "citations": [
            "partner_master.partner_id",
            "partner_master.partner_name",
            "partner_metrics.revenue"
        ]
    },

    "Q006": {
        "intent": "active_partners_with_pricing_exceptions",
        "example_question": "Which active partners have pricing exceptions?",
        "keywords": ["active", "partners", "pricing", "exceptions"],
        "answer_type": "Filtered list",
        "supported": True,
        "sql": """
            SELECT
                pm.partner_id,
                pm.partner_name,
                pm.partner_status,
                pp.pricing_tier,
                pp.exception_flag,
                pp.exception_type,
                pp.negotiated_bps,
                pp.approval_status
            FROM partner_master pm
            JOIN partner_pricing pp
                ON pm.partner_id = pp.partner_id
            WHERE pm.partner_status = 'Active'
              AND pp.exception_flag = 1
            ORDER BY pm.partner_name;
        """,
        "citations": [
            "partner_master.partner_id",
            "partner_master.partner_name",
            "partner_master.partner_status",
            "partner_pricing.pricing_tier",
            "partner_pricing.exception_flag",
            "partner_pricing.exception_type",
            "partner_pricing.negotiated_bps",
            "partner_pricing.approval_status"
        ]
    },

    "Q007": {
        "intent": "missing_demographic_information",
        "example_question": "Which partners have missing city or state information?",
        "keywords": ["missing", "city", "state", "demographic"],
        "answer_type": "Data quality list",
        "supported": True,
        "sql": """
            SELECT
                partner_id,
                partner_name,
                city,
                state,
                region
            FROM partner_master
            WHERE city IS NULL
               OR state IS NULL
               OR TRIM(city) = ''
               OR TRIM(state) = '';
        """,
        "citations": [
            "partner_master.partner_id",
            "partner_master.partner_name",
            "partner_master.city",
            "partner_master.state",
            "partner_master.region"
        ]
    },

    "Q008": {
        "intent": "average_growth_by_pricing_tier",
        "example_question": "Compare average transaction growth by pricing tier.",
        "keywords": ["average", "growth", "pricing", "tier", "compare"],
        "answer_type": "Comparison summary",
        "supported": True,
        "sql": """
            SELECT
                pp.pricing_tier,
                ROUND(AVG(mt.growth_pct), 2) AS avg_growth_pct,
                COUNT(*) AS metric_record_count
            FROM partner_pricing pp
            JOIN partner_metrics mt
                ON pp.partner_id = mt.partner_id
            WHERE mt.growth_pct IS NOT NULL
            GROUP BY pp.pricing_tier
            ORDER BY avg_growth_pct DESC;
        """,
        "citations": [
            "partner_pricing.pricing_tier",
            "partner_metrics.growth_pct"
        ]
    },

    "Q009": {
        "intent": "enterprise_partners_california",
        "example_question": "Which enterprise partners are in California?",
        "keywords": ["enterprise", "partners", "california", "ca"],
        "answer_type": "Filtered list",
        "supported": True,
        "sql": """
            SELECT
                partner_id,
                partner_name,
                partner_size,
                state,
                city,
                partner_status
            FROM partner_master
            WHERE partner_size = 'Enterprise'
              AND state = 'CA'
            ORDER BY partner_name;
        """,
        "citations": [
            "partner_master.partner_id",
            "partner_master.partner_name",
            "partner_master.partner_size",
            "partner_master.state",
            "partner_master.city",
            "partner_master.partner_status"
        ]
    },

    "Q010": {
        "intent": "declining_growth_latest_month",
        "example_question": "Which partners had declining transaction growth in the latest month?",
        "keywords": ["declining", "growth", "latest", "month"],
        "answer_type": "Filtered list",
        "supported": True,
        "sql": """
            WITH latest_month AS (
                SELECT MAX(date(transaction_month)) AS max_month
                FROM partner_metrics
            )
            SELECT
                pm.partner_id,
                pm.partner_name,
                mt.transaction_month,
                mt.growth_pct
            FROM partner_master pm
            JOIN partner_metrics mt
                ON pm.partner_id = mt.partner_id
            WHERE date(mt.transaction_month) = (SELECT max_month FROM latest_month)
              AND mt.growth_pct < 0
            ORDER BY mt.growth_pct ASC;
        """,
        "citations": [
            "partner_master.partner_id",
            "partner_master.partner_name",
            "partner_metrics.transaction_month",
            "partner_metrics.growth_pct"
        ]
    }
}

In [9]:
supported_question_rows = []

for qid, item in query_library.items():
    supported_question_rows.append({
        "question_id": qid,
        "intent": item["intent"],
        "example_question": item["example_question"],
        "answer_type": item["answer_type"],
        "supported": item["supported"],
        "citations": ", ".join(item["citations"])
    })

supported_question_registry = pd.DataFrame(supported_question_rows)

supported_question_registry.to_csv(
    os.path.join(phase6_path, "phase6_supported_question_registry.csv"),
    index=False
)

supported_question_registry

,question_id,intent,example_question,answer_type,supported,citations
0,Q001,technology_az_growth_above_20,Show me technology partners in Arizona with tr...,Filtered list,True,"partner_master.partner_id, partner_master.part..."
1,Q002,strategic_pricing_partners,Which partners are on the Strategic pricing tier?,Filtered list,True,"partner_master.partner_id, partner_master.part..."
2,Q003,pricing_reviews_due_90_days,Which partners have pricing reviews due in the...,Filtered list,True,"partner_master.partner_id, partner_master.part..."
3,Q004,highest_average_transaction_volume_by_industry,Which industry has the highest average transac...,Ranking summary,True,"partner_master.industry_vertical, partner_metr..."
4,Q005,top_5_partners_by_revenue,Show the top 5 partners by revenue.,Top 5 ranking,True,"partner_master.partner_id, partner_master.part..."
5,Q006,active_partners_with_pricing_exceptions,Which active partners have pricing exceptions?,Filtered list,True,"partner_master.partner_id, partner_master.part..."
6,Q007,missing_demographic_information,Which partners have missing city or state info...,Data quality list,True,"partner_master.partner_id, partner_master.part..."
7,Q008,average_growth_by_pricing_tier,Compare average transaction growth by pricing ...,Comparison summary,True,"partner_pricing.pricing_tier, partner_metrics...."
8,Q009,enterprise_partners_california,Which enterprise partners are in California?,Filtered list,True,"partner_master.partner_id, partner_master.part..."
9,Q010,declining_growth_latest_month,Which partners had declining transaction growt...,Filtered list,True,"partner_master.partner_id, partner_master.part..."


In [10]:
def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [11]:
def match_question_to_intent(user_question, minimum_score=2):
    normalized_question = normalize_text(user_question)

    best_match = None
    best_score = 0

    for qid, item in query_library.items():
        score = 0

        for keyword in item["keywords"]:
            keyword_normalized = normalize_text(keyword)

            if keyword_normalized in normalized_question:
                score += 1

        if score > best_score:
            best_score = score
            best_match = qid

    if best_match is None or best_score < minimum_score:
        return None, best_score

    return best_match, best_score

In [12]:
match_question_to_intent("Show me strategic pricing partners")

('Q002', 3)

In [13]:
match_question_to_intent("Predict next year revenue")

(None, 1)

In [14]:
def detect_unsupported_question(user_question):
    question = normalize_text(user_question)

    unsupported_rules = {
        "data_modification": [
            "update", "delete", "insert", "drop", "alter", "change pricing",
            "modify pricing", "approve pricing", "reject pricing", "create record"
        ],
        "forecasting": [
            "predict", "forecast", "next year", "next quarter", "future revenue",
            "future growth", "projection"
        ],
        "production_or_realtime": [
            "real time", "live data", "production", "current authorization",
            "today's transactions", "today transactions"
        ],
        "sensitive_data": [
            "ssn", "social security", "card number", "account number",
            "customer name", "personal information", "pii"
        ],
        "unsupported_external_data": [
            "market share", "competitor", "stock price", "news", "external"
        ]
    }

    for category, keywords in unsupported_rules.items():
        for keyword in keywords:
            if keyword in question:
                return True, category, keyword

    return False, None, None

In [15]:
detect_unsupported_question("Can you update pricing for partner P0001?")

(True, 'data_modification', 'update')

In [16]:
detect_unsupported_question("Forecast revenue for next quarter")

(True, 'forecasting', 'forecast')

In [17]:
detect_unsupported_question("Which partners are on Strategic pricing?")

(False, None, None)

In [18]:
def validate_sql_safety(sql_query, allowed_tables=None):
    if allowed_tables is None:
        allowed_tables = ["partner_master", "partner_pricing", "partner_metrics"]

    sql_clean = sql_query.strip().lower()

    blocked_keywords = [
        "insert", "update", "delete", "drop", "alter", "truncate",
        "create", "replace", "attach", "detach", "pragma", "vacuum"
    ]

    for keyword in blocked_keywords:
        pattern = r"\b" + keyword + r"\b"
        if re.search(pattern, sql_clean):
            return False, f"Blocked SQL keyword detected: {keyword}"

    first_word = sql_clean.split()[0]

    if first_word not in ["select", "with"]:
        return False, "Only SELECT or WITH queries are allowed."

    referenced_tables = re.findall(r"\bfrom\s+([a-zA-Z_][a-zA-Z0-9_]*)|\bjoin\s+([a-zA-Z_][a-zA-Z0-9_]*)", sql_clean)

    flat_tables = []

    for table_pair in referenced_tables:
        for table in table_pair:
            if table:
                flat_tables.append(table)

    for table in flat_tables:
        if table not in allowed_tables:
            return False, f"Unauthorized table referenced: {table}"

    return True, "SQL passed safety validation."

In [19]:
validate_sql_safety(query_library["Q001"]["sql"])

(True, 'SQL passed safety validation.')

In [20]:
validate_sql_safety("DROP TABLE partner_master;")

(False, 'Blocked SQL keyword detected: drop')

In [21]:
def audit_citations(question_id, citations, database_schema):
    audit_rows = []

    for citation in citations:
        if "." not in citation:
            audit_rows.append({
                "question_id": question_id,
                "citation": citation,
                "table_name": None,
                "field_name": None,
                "format_valid": False,
                "table_exists": False,
                "field_exists": False,
                "citation_valid": False,
                "issue": "Citation is not in table.field format"
            })
            continue

        table_name, field_name = citation.split(".", 1)

        table_exists = table_name in database_schema
        field_exists = table_exists and field_name in database_schema[table_name]

        citation_valid = table_exists and field_exists

        if not table_exists:
            issue = "Table does not exist"
        elif not field_exists:
            issue = "Field does not exist in table"
        else:
            issue = "Valid citation"

        audit_rows.append({
            "question_id": question_id,
            "citation": citation,
            "table_name": table_name,
            "field_name": field_name,
            "format_valid": True,
            "table_exists": table_exists,
            "field_exists": field_exists,
            "citation_valid": citation_valid,
            "issue": issue
        })

    return pd.DataFrame(audit_rows)

In [22]:
audit_citations(
    "Q001",
    query_library["Q001"]["citations"],
    database_schema
)

,question_id,citation,table_name,field_name,format_valid,table_exists,field_exists,citation_valid,issue
0,Q001,partner_master.partner_id,partner_master,partner_id,True,True,True,True,Valid citation
1,Q001,partner_master.partner_name,partner_master,partner_name,True,True,True,True,Valid citation
2,Q001,partner_master.partner_type,partner_master,partner_type,True,True,True,True,Valid citation
3,Q001,partner_master.industry_vertical,partner_master,industry_vertical,True,True,True,True,Valid citation
4,Q001,partner_master.state,partner_master,state,True,True,True,True,Valid citation
5,Q001,partner_metrics.transaction_month,partner_metrics,transaction_month,True,True,True,True,Valid citation
6,Q001,partner_metrics.growth_pct,partner_metrics,growth_pct,True,True,True,True,Valid citation


In [23]:
all_citation_audits = []

for qid, item in query_library.items():
    audit_df = audit_citations(
        question_id=qid,
        citations=item["citations"],
        database_schema=database_schema
    )

    all_citation_audits.append(audit_df)

citation_audit_results = pd.concat(all_citation_audits, ignore_index=True)

citation_audit_results.to_csv(
    os.path.join(phase6_path, "phase6_citation_audit_results.csv"),
    index=False
)

citation_audit_results

,question_id,citation,table_name,field_name,format_valid,table_exists,field_exists,citation_valid,issue
0,Q001,partner_master.partner_id,partner_master,partner_id,True,True,True,True,Valid citation
1,Q001,partner_master.partner_name,partner_master,partner_name,True,True,True,True,Valid citation
2,Q001,partner_master.partner_type,partner_master,partner_type,True,True,True,True,Valid citation
3,Q001,partner_master.industry_vertical,partner_master,industry_vertical,True,True,True,True,Valid citation
4,Q001,partner_master.state,partner_master,state,True,True,True,True,Valid citation
5,Q001,partner_metrics.transaction_month,partner_metrics,transaction_month,True,True,True,True,Valid citation
6,Q001,partner_metrics.growth_pct,partner_metrics,growth_pct,True,True,True,True,Valid citation
7,Q002,partner_master.partner_id,partner_master,partner_id,True,True,True,True,Valid citation
8,Q002,partner_master.partner_name,partner_master,partner_name,True,True,True,True,Valid citation
9,Q002,partner_master.partner_status,partner_master,partner_status,True,True,True,True,Valid citation


In [24]:
total_citations = len(citation_audit_results)
valid_citations = citation_audit_results["citation_valid"].sum()

citation_accuracy = valid_citations / total_citations if total_citations > 0 else 0

print("Total citations:", total_citations)
print("Valid citations:", valid_citations)
print("Citation accuracy:", round(citation_accuracy * 100, 2), "%")

Total citations: 49
Valid citations: 49
Citation accuracy: 100.0 %


In [25]:
def generate_answer_summary(question_id, result_df):
    row_count = len(result_df)

    if row_count == 0:
        return "No matching records were found in the available PartnerLens dataset."

    if question_id == "Q001":
        return f"I found {row_count} Technology partner metric records in Arizona with transaction growth above 20%."

    elif question_id == "Q002":
        return f"I found {row_count} partners currently assigned to the Strategic pricing tier."

    elif question_id == "Q003":
        return f"I found {row_count} partners with pricing reviews due in the next 90 days."

    elif question_id == "Q004":
        top_industry = result_df.iloc[0]["industry_vertical"]
        top_volume = result_df.iloc[0]["avg_transaction_volume"]
        return f"The industry with the highest average transaction volume is {top_industry}, with an average transaction volume of ${top_volume:,.2f}."

    elif question_id == "Q005":
        top_partner = result_df.iloc[0]["partner_name"]
        top_revenue = result_df.iloc[0]["total_revenue"]
        return f"The top revenue-generating partner is {top_partner}, with total revenue of ${top_revenue:,.2f}."

    elif question_id == "Q006":
        return f"I found {row_count} active partners with pricing exceptions."

    elif question_id == "Q007":
        return f"I found {row_count} partners with missing city or state information."

    elif question_id == "Q008":
        top_tier = result_df.iloc[0]["pricing_tier"]
        top_growth = result_df.iloc[0]["avg_growth_pct"]
        return f"The pricing tier with the highest average transaction growth is {top_tier}, with average growth of {top_growth}%."

    elif question_id == "Q009":
        return f"I found {row_count} enterprise partners located in California."

    elif question_id == "Q010":
        return f"I found {row_count} partners with declining transaction growth in the latest month."

    else:
        return f"The query returned {row_count} records."

In [26]:
phase6_audit_logs = []

def partnerlens_guarded_assistant(user_question, show_sql=True, show_citations=True, preview_rows=10):
    timestamp = datetime.now()

    unsupported, unsupported_category, unsupported_keyword = detect_unsupported_question(user_question)

    if unsupported:
        answer = (
            "This question is outside the supported scope of the PartnerLens Copilot MVP. "
            f"Reason: {unsupported_category}. "
            "The current prototype only supports read-only partner demographic, pricing, and monthly performance questions."
        )

        log_row = {
            "timestamp": timestamp,
            "user_question": user_question,
            "matched_question_id": None,
            "matched_intent": None,
            "match_score": 0,
            "row_count": 0,
            "sql_safe": None,
            "citation_accuracy": None,
            "status": "Unsupported question",
            "unsupported_category": unsupported_category,
            "unsupported_keyword": unsupported_keyword
        }

        phase6_audit_logs.append(log_row)

        print("User Question:")
        print(user_question)
        print("\nAnswer:")
        print(answer)

        return {
            "answer": answer,
            "result": pd.DataFrame(),
            "citations": [],
            "sql": None,
            "status": "Unsupported question"
        }

    matched_qid, match_score = match_question_to_intent(user_question, minimum_score=2)

    if matched_qid is None:
        answer = (
            "I could not confidently match this question to the supported PartnerLens query library. "
            "Please ask a question about partner demographics, pricing tiers, pricing reviews, revenue, transaction volume, growth, or pricing exceptions."
        )

        log_row = {
            "timestamp": timestamp,
            "user_question": user_question,
            "matched_question_id": None,
            "matched_intent": None,
            "match_score": match_score,
            "row_count": 0,
            "sql_safe": None,
            "citation_accuracy": None,
            "status": "No confident match",
            "unsupported_category": None,
            "unsupported_keyword": None
        }

        phase6_audit_logs.append(log_row)

        print("User Question:")
        print(user_question)
        print("\nAnswer:")
        print(answer)

        return {
            "answer": answer,
            "result": pd.DataFrame(),
            "citations": [],
            "sql": None,
            "status": "No confident match"
        }

    query_item = query_library[matched_qid]
    sql_query = query_item["sql"]
    citations = query_item["citations"]

    sql_safe, sql_safety_message = validate_sql_safety(sql_query)

    if not sql_safe:
        answer = f"The generated SQL was blocked by safety validation. Reason: {sql_safety_message}"

        log_row = {
            "timestamp": timestamp,
            "user_question": user_question,
            "matched_question_id": matched_qid,
            "matched_intent": query_item["intent"],
            "match_score": match_score,
            "row_count": 0,
            "sql_safe": False,
            "citation_accuracy": None,
            "status": "SQL blocked",
            "unsupported_category": None,
            "unsupported_keyword": None
        }

        phase6_audit_logs.append(log_row)

        print("User Question:")
        print(user_question)
        print("\nAnswer:")
        print(answer)

        return {
            "answer": answer,
            "result": pd.DataFrame(),
            "citations": citations,
            "sql": sql_query,
            "status": "SQL blocked"
        }

    citation_audit_df = audit_citations(
        question_id=matched_qid,
        citations=citations,
        database_schema=database_schema
    )

    citation_valid_count = citation_audit_df["citation_valid"].sum()
    citation_total_count = len(citation_audit_df)
    response_citation_accuracy = citation_valid_count / citation_total_count if citation_total_count > 0 else 0

    if response_citation_accuracy < 1:
        answer = "The answer was blocked because one or more source citations failed validation."

        log_row = {
            "timestamp": timestamp,
            "user_question": user_question,
            "matched_question_id": matched_qid,
            "matched_intent": query_item["intent"],
            "match_score": match_score,
            "row_count": 0,
            "sql_safe": True,
            "citation_accuracy": response_citation_accuracy,
            "status": "Citation audit failed",
            "unsupported_category": None,
            "unsupported_keyword": None
        }

        phase6_audit_logs.append(log_row)

        print("User Question:")
        print(user_question)
        print("\nAnswer:")
        print(answer)
        display(citation_audit_df)

        return {
            "answer": answer,
            "result": pd.DataFrame(),
            "citations": citations,
            "sql": sql_query,
            "status": "Citation audit failed"
        }

    result_df = run_sql(sql_query)
    answer = generate_answer_summary(matched_qid, result_df)

    log_row = {
        "timestamp": timestamp,
        "user_question": user_question,
        "matched_question_id": matched_qid,
        "matched_intent": query_item["intent"],
        "match_score": match_score,
        "row_count": len(result_df),
        "sql_safe": True,
        "citation_accuracy": response_citation_accuracy,
        "status": "Success",
        "unsupported_category": None,
        "unsupported_keyword": None
    }

    phase6_audit_logs.append(log_row)

    print("User Question:")
    print(user_question)

    print("\nMatched Intent:")
    print(query_item["intent"])

    print("\nAnswer:")
    print(answer)

    print("\nAudit Status:")
    print("SQL safety:", sql_safety_message)
    print("Citation accuracy:", round(response_citation_accuracy * 100, 2), "%")

    if show_sql:
        print("\nSQL Used:")
        print(sql_query)

    if show_citations:
        print("\nSources Used:")
        for citation in citations:
            print("-", citation)

    print("\nResult Preview:")
    display(result_df.head(preview_rows))

    return {
        "answer": answer,
        "result": result_df,
        "citations": citations,
        "sql": sql_query,
        "status": "Success",
        "citation_audit": citation_audit_df
    }

In [27]:
partnerlens_guarded_assistant(
    "Show me technology partners in Arizona with transaction growth above 20%."
)

User Question:
Show me technology partners in Arizona with transaction growth above 20%.

Matched Intent:
technology_az_growth_above_20

Answer:
I found 85 Technology partner metric records in Arizona with transaction growth above 20%.

Audit Status:
SQL safety: SQL passed safety validation.
Citation accuracy: 100.0 %

SQL Used:

            SELECT
                pm.partner_id,
                pm.partner_name,
                pm.partner_type,
                pm.industry_vertical,
                pm.state,
                pm.partner_status,
                mt.transaction_month,
                mt.growth_pct
            FROM partner_master pm
            JOIN partner_metrics mt
                ON pm.partner_id = mt.partner_id
            WHERE pm.partner_type LIKE '%Technology%'
              AND pm.state = 'AZ'
              AND mt.growth_pct > 20
            ORDER BY mt.growth_pct DESC;
        

Sources Used:
- partner_master.partner_id
- partner_master.partner_name
- partner_master.

,partner_id,partner_name,partner_type,industry_vertical,state,partner_status,transaction_month,growth_pct
0,P004892,Summit Connect 4892,Technology Platform,Travel,AZ,Active,2024-12-01,55.72
1,P003960,Clearwater Processing 3960,Technology Platform,Hospitality,AZ,Pilot,2024-12-01,43.32
2,P004952,Granite Processing 4952,Technology Platform,Travel,AZ,Active,2024-11-01,41.59
3,P004612,Sunrise Connect 4612,Technology Platform,Travel,AZ,Active,2025-11-01,41.22
4,P004952,Granite Processing 4952,Technology Platform,Travel,AZ,Active,2025-12-01,40.97
5,P002009,Harbor Solutions 2009,Technology Platform,SaaS,AZ,Active,2024-02-01,37.18
6,P002733,Sunrise Market 2733,Technology Platform,Retail,AZ,Suspended,2025-10-01,36.61
7,P001717,Oakstone Gateway 1717,Technology Platform,Travel,AZ,Active,2024-11-01,36.24
8,P001160,Granite Processing 1160,Technology Platform,Hospitality,AZ,Active,2024-12-01,35.97
9,P003711,Granite Gateway 3711,Technology Platform,Travel,AZ,Active,2025-11-01,34.95


{'answer': 'I found 85 Technology partner metric records in Arizona with transaction growth above 20%.',
 'result':    partner_id                      partner_name         partner_type  \
 0     P004892               Summit Connect 4892  Technology Platform   
 1     P003960        Clearwater Processing 3960  Technology Platform   
 2     P004952           Granite Processing 4952  Technology Platform   
 3     P004612              Sunrise Connect 4612  Technology Platform   
 4     P004952           Granite Processing 4952  Technology Platform   
 ..        ...                               ...                  ...   
 80    P001423              RedRock Gateway 1423  Technology Platform   
 81    P001747  Riverbend Merchant Services 1747  Technology Platform   
 82    P002912   Oakstone Merchant Services 2912  Technology Platform   
 83    P001717             Oakstone Gateway 1717  Technology Platform   
 84    P003401         Silverline Solutions 3401  Technology Platform   
 
       

In [28]:
partnerlens_guarded_assistant(
    "Which partners are on the Strategic pricing tier?"
)

User Question:
Which partners are on the Strategic pricing tier?

Matched Intent:
strategic_pricing_partners

Answer:
I found 391 partners currently assigned to the Strategic pricing tier.

Audit Status:
SQL safety: SQL passed safety validation.
Citation accuracy: 100.0 %

SQL Used:

            SELECT
                pm.partner_id,
                pm.partner_name,
                pm.partner_status,
                pp.pricing_tier,
                pp.recommended_pricing_plan_id,
                pp.approval_status
            FROM partner_master pm
            JOIN partner_pricing pp
                ON pm.partner_id = pp.partner_id
            WHERE pp.pricing_tier = 'STRATEGIC'
            ORDER BY pm.partner_name;
        

Sources Used:
- partner_master.partner_id
- partner_master.partner_name
- partner_master.partner_status
- partner_pricing.pricing_tier
- partner_pricing.recommended_pricing_plan_id
- partner_pricing.approval_status

Result Preview:


,partner_id,partner_name,partner_status,pricing_tier,recommended_pricing_plan_id,approval_status
0,P000760,BluePeak Billing 0760,Active,STRATEGIC,STRATEGIC,Not Required
1,P002523,BluePeak Billing 2523,Active,STRATEGIC,STRATEGIC,Not Required
2,P004400,BluePeak Billing 4400,Active,STRATEGIC,STRATEGIC,Not Required
3,P004791,BluePeak Billing 4791,Active,STRATEGIC,STRATEGIC,Not Required
4,P001740,BluePeak Checkout 1740,Active,STRATEGIC,STRATEGIC,Not Required
5,P004440,BluePeak Commerce 4440,Active,STRATEGIC,STRATEGIC,Not Required
6,P004673,BluePeak Commerce 4673,Active,STRATEGIC,GROWTH,Approved
7,P002564,BluePeak Financial 2564,Active,STRATEGIC,STRATEGIC,Not Required
8,P000116,BluePeak Gateway 0116,Active,STRATEGIC,STRATEGIC,Not Required
9,P000462,BluePeak Gateway 0462,Active,STRATEGIC,STRATEGIC,Not Required


{'answer': 'I found 391 partners currently assigned to the Strategic pricing tier.',
 'result':     partner_id             partner_name partner_status pricing_tier  \
 0      P000760    BluePeak Billing 0760         Active    STRATEGIC   
 1      P002523    BluePeak Billing 2523         Active    STRATEGIC   
 2      P004400    BluePeak Billing 4400         Active    STRATEGIC   
 3      P004791    BluePeak Billing 4791         Active    STRATEGIC   
 4      P001740   BluePeak Checkout 1740         Active    STRATEGIC   
 ..         ...                      ...            ...          ...   
 386    P004232    Sunrise Platform 4232         Active    STRATEGIC   
 387    P004851    Sunrise Platform 4851         Active    STRATEGIC   
 388    P002285  Sunrise Processing 2285         Active    STRATEGIC   
 389    P002432  Sunrise Processing 2432         Active    STRATEGIC   
 390    P002732  Sunrise Processing 2732         Active    STRATEGIC   
 
     recommended_pricing_plan_id approv

In [29]:
partnerlens_guarded_assistant(
    "Which partners have pricing reviews due in the next 90 days?"
)

User Question:
Which partners have pricing reviews due in the next 90 days?

Matched Intent:
pricing_reviews_due_90_days

Answer:
I found 361 partners with pricing reviews due in the next 90 days.

Audit Status:
SQL safety: SQL passed safety validation.
Citation accuracy: 100.0 %

SQL Used:

            SELECT
                pm.partner_id,
                pm.partner_name,
                pp.pricing_tier,
                pp.approval_status,
                pp.review_due_date,
                pp.contract_status
            FROM partner_master pm
            JOIN partner_pricing pp
                ON pm.partner_id = pp.partner_id
            WHERE date(pp.review_due_date) BETWEEN date('2026-06-28') AND date('2026-09-26')
            ORDER BY date(pp.review_due_date);
        

Sources Used:
- partner_master.partner_id
- partner_master.partner_name
- partner_pricing.pricing_tier
- partner_pricing.approval_status
- partner_pricing.review_due_date
- partner_pricing.contract_status

Result P

,partner_id,partner_name,pricing_tier,approval_status,review_due_date,contract_status
0,P000170,Keystone PayWorks 0170,LAUNCH,Not Required,2026-06-28,Review Due Soon
1,P000368,Skyline Market 0368,PLATFORM_ISV,Not Required,2026-06-28,Review Due Soon
2,P003504,Cedar PayWorks 3504,GROWTH,Not Required,2026-06-28,Review Due Soon
3,P003984,Evergreen Financial 3984,LAUNCH,Not Required,2026-06-28,Review Due Soon
4,P004509,Prairie Payments 4509,PROMO_GROWTH,Approved,2026-06-28,Review Due Soon
5,P004928,Cedar Solutions 4928,LAUNCH,Not Required,2026-06-28,Review Due Soon
6,P000980,Clearwater Merchant Services 0980,SCALE,Approved,2026-06-29,Review Due Soon
7,P001302,Summit Market 1302,GROWTH,Not Required,2026-06-29,Review Due Soon
8,P001751,Cedar Market 1751,PLATFORM_ISV,Not Required,2026-06-29,Review Due Soon
9,P002144,BluePeak Connect 2144,HIGH_RISK,Not Required,2026-06-29,Review Due Soon


{'answer': 'I found 361 partners with pricing reviews due in the next 90 days.',
 'result':     partner_id              partner_name  pricing_tier approval_status  \
 0      P000170    Keystone PayWorks 0170        LAUNCH    Not Required   
 1      P000368       Skyline Market 0368  PLATFORM_ISV    Not Required   
 2      P003504       Cedar PayWorks 3504        GROWTH    Not Required   
 3      P003984  Evergreen Financial 3984        LAUNCH    Not Required   
 4      P004509     Prairie Payments 4509  PROMO_GROWTH        Approved   
 ..         ...                       ...           ...             ...   
 356    P002608     Granite Commerce 2608  PLATFORM_ISV    Not Required   
 357    P002894    NorthStar Gateway 2894     HIGH_RISK    Not Required   
 358    P002921     RedRock Payments 2921        LAUNCH    Not Required   
 359    P003131     Skyline PayWorks 3131        GROWTH    Not Required   
 360    P003372   Silverline Gateway 3372        GROWTH    Not Required   
 
     re

In [30]:
partnerlens_guarded_assistant(
    "Show the top 5 partners by revenue."
)

User Question:
Show the top 5 partners by revenue.

Matched Intent:
top_5_partners_by_revenue

Answer:
The top revenue-generating partner is Skyline Payments 0610, with total revenue of $64,847,292.41.

Audit Status:
SQL safety: SQL passed safety validation.
Citation accuracy: 100.0 %

SQL Used:

            SELECT
                pm.partner_id,
                pm.partner_name,
                ROUND(SUM(mt.revenue), 2) AS total_revenue
            FROM partner_master pm
            JOIN partner_metrics mt
                ON pm.partner_id = mt.partner_id
            GROUP BY pm.partner_id, pm.partner_name
            ORDER BY total_revenue DESC
            LIMIT 5;
        

Sources Used:
- partner_master.partner_id
- partner_master.partner_name
- partner_metrics.revenue

Result Preview:


,partner_id,partner_name,total_revenue
0,P000610,Skyline Payments 0610,64847292.41
1,P002651,RedRock PayTech 2651,28539531.11
2,P003122,Cedar Market 3122,27485493.95
3,P004602,NorthStar Commerce 4602,26392278.10
4,P001422,Brightpath Platform 1422,25564051.57


{'answer': 'The top revenue-generating partner is Skyline Payments 0610, with total revenue of $64,847,292.41.',
 'result':   partner_id              partner_name  total_revenue
 0    P000610     Skyline Payments 0610    64847292.41
 1    P002651      RedRock PayTech 2651    28539531.11
 2    P003122         Cedar Market 3122    27485493.95
 3    P004602   NorthStar Commerce 4602    26392278.10
 4    P001422  Brightpath Platform 1422    25564051.57,
 'citations': ['partner_master.partner_id',
  'partner_master.partner_name',
  'partner_metrics.revenue'],
 'sql': '\n            SELECT\n                pm.partner_id,\n                pm.partner_name,\n                ROUND(SUM(mt.revenue), 2) AS total_revenue\n            FROM partner_master pm\n            JOIN partner_metrics mt\n                ON pm.partner_id = mt.partner_id\n            GROUP BY pm.partner_id, pm.partner_name\n            ORDER BY total_revenue DESC\n            LIMIT 5;\n        ',
 'status': 'Success',
 'citation

In [31]:
partnerlens_guarded_assistant(
    "Compare average transaction growth by pricing tier."
)

User Question:
Compare average transaction growth by pricing tier.

Matched Intent:
average_growth_by_pricing_tier

Answer:
The pricing tier with the highest average transaction growth is PLATFORM_ISV, with average growth of 2.97%.

Audit Status:
SQL safety: SQL passed safety validation.
Citation accuracy: 100.0 %

SQL Used:

            SELECT
                pp.pricing_tier,
                ROUND(AVG(mt.growth_pct), 2) AS avg_growth_pct,
                COUNT(*) AS metric_record_count
            FROM partner_pricing pp
            JOIN partner_metrics mt
                ON pp.partner_id = mt.partner_id
            WHERE mt.growth_pct IS NOT NULL
            GROUP BY pp.pricing_tier
            ORDER BY avg_growth_pct DESC;
        

Sources Used:
- partner_pricing.pricing_tier
- partner_metrics.growth_pct

Result Preview:


,pricing_tier,avg_growth_pct,metric_record_count
0,PLATFORM_ISV,2.97,16399
1,LEGACY_STANDARD,2.77,2967
2,STRATEGIC,2.44,8993
3,HIGH_RISK,2.42,19803
4,SCALE,2.40,13662
5,GROWTH,2.09,24219
6,PROMO_GROWTH,2.00,943
7,LEGACY_ENTERPRISE,2.00,3197
8,LAUNCH,1.36,24817


{'answer': 'The pricing tier with the highest average transaction growth is PLATFORM_ISV, with average growth of 2.97%.',
 'result':         pricing_tier  avg_growth_pct  metric_record_count
 0       PLATFORM_ISV            2.97                16399
 1    LEGACY_STANDARD            2.77                 2967
 2          STRATEGIC            2.44                 8993
 3          HIGH_RISK            2.42                19803
 4              SCALE            2.40                13662
 5             GROWTH            2.09                24219
 6       PROMO_GROWTH            2.00                  943
 7  LEGACY_ENTERPRISE            2.00                 3197
 8             LAUNCH            1.36                24817,
 'citations': ['partner_pricing.pricing_tier', 'partner_metrics.growth_pct'],
 'sql': '\n            SELECT\n                pp.pricing_tier,\n                ROUND(AVG(mt.growth_pct), 2) AS avg_growth_pct,\n                COUNT(*) AS metric_record_count\n            FROM par

In [32]:
unsupported_test_questions = [
    "Can you update pricing for partner P0001?",
    "Delete inactive partners from the database.",
    "Forecast next quarter revenue by partner.",
    "Show customer card numbers for each partner.",
    "Pull real-time production transaction data.",
    "What is the stock price of our largest partner?",
    "Approve all pending pricing exceptions."
]

for question in unsupported_test_questions:
    print("=" * 100)
    partnerlens_guarded_assistant(question, show_sql=False, show_citations=False)

User Question:
Can you update pricing for partner P0001?

Answer:
This question is outside the supported scope of the PartnerLens Copilot MVP. Reason: data_modification. The current prototype only supports read-only partner demographic, pricing, and monthly performance questions.
User Question:
Delete inactive partners from the database.

Answer:
This question is outside the supported scope of the PartnerLens Copilot MVP. Reason: data_modification. The current prototype only supports read-only partner demographic, pricing, and monthly performance questions.
User Question:
Forecast next quarter revenue by partner.

Answer:
This question is outside the supported scope of the PartnerLens Copilot MVP. Reason: forecasting. The current prototype only supports read-only partner demographic, pricing, and monthly performance questions.
User Question:
Show customer card numbers for each partner.

Answer:
This question is outside the supported scope of the PartnerLens Copilot MVP. Reason: sensiti

,partner_id,partner_name,partner_status,pricing_tier,exception_flag,exception_type,negotiated_bps,approval_status
0,P000573,BluePeak Billing 0573,Active,LEGACY_STANDARD,1,Legacy plan extension,53.65,Approved
1,P000735,BluePeak Billing 0735,Active,PLATFORM_ISV,1,High-risk waiver,55.48,Approved
2,P004415,BluePeak Billing 4415,Active,GROWTH,1,Volume threshold waiver,52.99,Pending
3,P002322,BluePeak Checkout 2322,Active,LEGACY_STANDARD,1,Waived monthly minimum,67.65,Pending
4,P002996,BluePeak Checkout 2996,Active,SCALE,1,High-risk waiver,48.18,Approved
5,P004673,BluePeak Commerce 4673,Active,STRATEGIC,1,Legacy plan extension,15.39,Approved
6,P001468,BluePeak Connect 1468,Active,HIGH_RISK,1,High-risk waiver,113.58,Approved
7,P004552,BluePeak Connect 4552,Active,LAUNCH,1,Legacy plan extension,61.46,Approved
8,P004675,BluePeak Connect 4675,Active,LEGACY_STANDARD,1,High-risk waiver,73.08,Pending
9,P000862,BluePeak Financial 0862,Active,GROWTH,1,Waived monthly minimum,70.61,Approved


In [33]:
guardrail_test_results = []

test_questions = [
    {
        "question": "Show me technology partners in Arizona with transaction growth above 20%.",
        "expected_status": "Success"
    },
    {
        "question": "Which partners are on the Strategic pricing tier?",
        "expected_status": "Success"
    },
    {
        "question": "Can you update pricing for partner P0001?",
        "expected_status": "Unsupported question"
    },
    {
        "question": "Delete inactive partners from the database.",
        "expected_status": "Unsupported question"
    },
    {
        "question": "Forecast next quarter revenue by partner.",
        "expected_status": "Unsupported question"
    },
    {
        "question": "Show customer card numbers for each partner.",
        "expected_status": "Unsupported question"
    },
    {
        "question": "Tell me something interesting about partners.",
        "expected_status": "No confident match"
    }
]

for item in test_questions:
    response = partnerlens_guarded_assistant(
        item["question"],
        show_sql=False,
        show_citations=False,
        preview_rows=5
    )

    actual_status = response["status"]

    guardrail_test_results.append({
        "question": item["question"],
        "expected_status": item["expected_status"],
        "actual_status": actual_status,
        "test_passed": item["expected_status"] == actual_status
    })

guardrail_test_results_df = pd.DataFrame(guardrail_test_results)

guardrail_test_results_df.to_csv(
    os.path.join(phase6_path, "phase6_guardrail_test_results.csv"),
    index=False
)

guardrail_test_results_df

User Question:
Show me technology partners in Arizona with transaction growth above 20%.

Matched Intent:
technology_az_growth_above_20

Answer:
I found 85 Technology partner metric records in Arizona with transaction growth above 20%.

Audit Status:
SQL safety: SQL passed safety validation.
Citation accuracy: 100.0 %

Result Preview:


,partner_id,partner_name,partner_type,industry_vertical,state,partner_status,transaction_month,growth_pct
0,P004892,Summit Connect 4892,Technology Platform,Travel,AZ,Active,2024-12-01,55.72
1,P003960,Clearwater Processing 3960,Technology Platform,Hospitality,AZ,Pilot,2024-12-01,43.32
2,P004952,Granite Processing 4952,Technology Platform,Travel,AZ,Active,2024-11-01,41.59
3,P004612,Sunrise Connect 4612,Technology Platform,Travel,AZ,Active,2025-11-01,41.22
4,P004952,Granite Processing 4952,Technology Platform,Travel,AZ,Active,2025-12-01,40.97


User Question:
Which partners are on the Strategic pricing tier?

Matched Intent:
strategic_pricing_partners

Answer:
I found 391 partners currently assigned to the Strategic pricing tier.

Audit Status:
SQL safety: SQL passed safety validation.
Citation accuracy: 100.0 %

Result Preview:


,partner_id,partner_name,partner_status,pricing_tier,recommended_pricing_plan_id,approval_status
0,P000760,BluePeak Billing 0760,Active,STRATEGIC,STRATEGIC,Not Required
1,P002523,BluePeak Billing 2523,Active,STRATEGIC,STRATEGIC,Not Required
2,P004400,BluePeak Billing 4400,Active,STRATEGIC,STRATEGIC,Not Required
3,P004791,BluePeak Billing 4791,Active,STRATEGIC,STRATEGIC,Not Required
4,P001740,BluePeak Checkout 1740,Active,STRATEGIC,STRATEGIC,Not Required


User Question:
Can you update pricing for partner P0001?

Answer:
This question is outside the supported scope of the PartnerLens Copilot MVP. Reason: data_modification. The current prototype only supports read-only partner demographic, pricing, and monthly performance questions.
User Question:
Delete inactive partners from the database.

Answer:
This question is outside the supported scope of the PartnerLens Copilot MVP. Reason: data_modification. The current prototype only supports read-only partner demographic, pricing, and monthly performance questions.
User Question:
Forecast next quarter revenue by partner.

Answer:
This question is outside the supported scope of the PartnerLens Copilot MVP. Reason: forecasting. The current prototype only supports read-only partner demographic, pricing, and monthly performance questions.
User Question:
Show customer card numbers for each partner.

Answer:
This question is outside the supported scope of the PartnerLens Copilot MVP. Reason: sensiti

,question,expected_status,actual_status,test_passed
0,Show me technology partners in Arizona with tr...,Success,Success,True
1,Which partners are on the Strategic pricing tier?,Success,Success,True
2,Can you update pricing for partner P0001?,Unsupported question,Unsupported question,True
3,Delete inactive partners from the database.,Unsupported question,Unsupported question,True
4,Forecast next quarter revenue by partner.,Unsupported question,Unsupported question,True
5,Show customer card numbers for each partner.,Unsupported question,Unsupported question,True
6,Tell me something interesting about partners.,No confident match,No confident match,True


In [34]:
phase6_audit_logs_df = pd.DataFrame(phase6_audit_logs)

phase6_audit_logs_df.to_csv(
    os.path.join(phase6_path, "phase6_assistant_audit_logs.csv"),
    index=False
)

phase6_audit_logs_df.head()

,timestamp,user_question,matched_question_id,matched_intent,match_score,row_count,sql_safe,citation_accuracy,status,unsupported_category,unsupported_keyword
0,2026-07-05 00:47:41.290414,Show me technology partners in Arizona with tr...,Q001,technology_az_growth_above_20,4,85,True,1.0,Success,None,None
1,2026-07-05 00:47:50.392087,Which partners are on the Strategic pricing tier?,Q002,strategic_pricing_partners,4,391,True,1.0,Success,None,None
2,2026-07-05 00:48:14.534517,Which partners have pricing reviews due in the...,Q003,pricing_reviews_due_90_days,5,361,True,1.0,Success,None,None
3,2026-07-05 00:48:20.869909,Show the top 5 partners by revenue.,Q005,top_5_partners_by_revenue,4,5,True,1.0,Success,None,None
4,2026-07-05 00:48:30.828590,Compare average transaction growth by pricing ...,Q008,average_growth_by_pricing_tier,5,9,True,1.0,Success,None,None


In [35]:
total_guardrail_tests = len(guardrail_test_results_df)
passed_guardrail_tests = guardrail_test_results_df["test_passed"].sum()

guardrail_pass_rate = passed_guardrail_tests / total_guardrail_tests if total_guardrail_tests > 0 else 0

total_citations = len(citation_audit_results)
valid_citations = citation_audit_results["citation_valid"].sum()

citation_accuracy = valid_citations / total_citations if total_citations > 0 else 0

print("Guardrail tests:", total_guardrail_tests)
print("Passed guardrail tests:", passed_guardrail_tests)
print("Guardrail pass rate:", round(guardrail_pass_rate * 100, 2), "%")

print("\nTotal citations:", total_citations)
print("Valid citations:", valid_citations)
print("Citation accuracy:", round(citation_accuracy * 100, 2), "%")

Guardrail tests: 7
Passed guardrail tests: 7
Guardrail pass rate: 100.0 %

Total citations: 49
Valid citations: 49
Citation accuracy: 100.0 %


In [36]:
summary_text = f"""
Phase 6 Summary: Citation Auditor, Guardrails, and Unsupported Question Handling

Phase 6 enhanced the PartnerLens Copilot prototype with governance, auditability, and safety controls.

The following capabilities were added:
1. Database schema registry for validating source-field citations.
2. Citation auditor that checks whether every citation follows table.field format and maps to a real SQLite table and column.
3. SQL safety validator that allows only read-only SELECT or WITH queries.
4. Unsupported-question detector for out-of-scope requests such as data modification, forecasting, production data access, sensitive information, and external market data.
5. Improved intent-matching threshold to reduce false-positive matches.
6. Guarded assistant response flow with audit status, SQL safety status, citation accuracy, and source-field citations.
7. Audit logging for supported, unsupported, blocked, and unmatched questions.

Phase 6 Results:
- Total citation records checked: {total_citations}
- Valid citation records: {valid_citations}
- Citation accuracy: {round(citation_accuracy * 100, 2)}%
- Total guardrail tests: {total_guardrail_tests}
- Passed guardrail tests: {passed_guardrail_tests}
- Guardrail pass rate: {round(guardrail_pass_rate * 100, 2)}%

This phase improves the trustworthiness of PartnerLens Copilot by ensuring that answers are backed by validated source fields, SQL execution is restricted to safe read-only queries, and unsupported requests are clearly rejected instead of producing unreliable or unsafe answers.
"""

summary_file = os.path.join(phase6_path, "phase6_summary.txt")

with open(summary_file, "w") as f:
    f.write(summary_text)

print(summary_text)
print("Summary saved to:", summary_file)


Phase 6 Summary: Citation Auditor, Guardrails, and Unsupported Question Handling

Phase 6 enhanced the PartnerLens Copilot prototype with governance, auditability, and safety controls.

The following capabilities were added:
1. Database schema registry for validating source-field citations.
2. Citation auditor that checks whether every citation follows table.field format and maps to a real SQLite table and column.
3. SQL safety validator that allows only read-only SELECT or WITH queries.
4. Unsupported-question detector for out-of-scope requests such as data modification, forecasting, production data access, sensitive information, and external market data.
5. Improved intent-matching threshold to reduce false-positive matches.
6. Guarded assistant response flow with audit status, SQL safety status, citation accuracy, and source-field citations.
7. Audit logging for supported, unsupported, blocked, and unmatched questions.

Phase 6 Results:
- Total citation records checked: 49
- Valid 

In [37]:
conn.close()
print("SQLite connection closed.")

SQLite connection closed.
